# Classifier Evaluation Lab

* Copy&paste your model for homework5 model
* Add grid search and train
* Compare performance
* Which one is better? Explain?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/log_reg/employee-turnover-balanced.csv')
df.head()

,left_company,age,frequency_of_travel,department,commuting_distance,education,satisfaction_with_environment,gender,seniority_level,position,satisfaction_with_job,married_or_single,last_raise_pct,last_performance_rating,total_years_working,years_at_company,years_in_current_job,years_since_last_promotion,years_with_current_supervisor
0,No,37,Travel_Rarely,Sales,16,4,4,Male,2,Sales Executive,3,Divorced,19,3,9,1,0,0,0
1,No,39,Travel_Rarely,Research & Development,3,2,3,Male,2,Laboratory Technician,3,Divorced,15,3,11,10,8,0,7
2,No,52,Travel_Frequently,Research & Development,25,4,3,Female,4,Manufacturing Director,4,Married,22,4,31,9,8,0,0
3,No,50,Non-Travel,Sales,1,3,4,Female,2,Sales Executive,3,Married,12,3,19,18,7,0,13
4,No,44,Travel_Rarely,Research & Development,4,3,4,Male,2,Healthcare Representative,2,Single,12,3,10,5,2,2,3


In [2]:
#model from week5
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

num_vars = ['age', 'commuting_distance', 'education', 'satisfaction_with_environment', 'seniority_level', 'satisfaction_with_job', 'last_raise_pct', 'last_performance_rating', 'total_years_working', 'years_at_company', 'years_in_current_job', 'years_since_last_promotion', 'years_with_current_supervisor']
cat_vars = ['frequency_of_travel', 'department', 'gender','position','married_or_single']

num_vars = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
num_df = df[num_vars]
corr_matrix = num_df.corr()
correlated_vars = set()
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) >0.7:
            col_i = corr_matrix.columns[i]
            col_j = corr_matrix.columns[j]
            correlated_vars.add(col_i)
            correlated_vars.add(col_j)
num_vars = [col for col in num_vars if col not in correlated_vars]

X = df.drop('left_company', axis=1)
y = df['left_company']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=124)


num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_vars),
    ('cat', cat_transformer, cat_vars)
])

p = Pipeline([
    ('preprocessor', preprocessor), 
    ('model', LogisticRegression(max_iter=10000))
])


In [3]:
##Applying gridSearch and Training

# Define a set of hyperparameters for grid search
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear', 'saga']
}

grid_search = GridSearchCV(p, param_grid, cv=5, scoring='accuracy', return_train_score=True)

#Training
grid_search.fit(X_train, y_train)

print("Best parameters: ", grid_search.best_params_)

Best parameters:  {'model__C': 1, 'model__penalty': 'l2', 'model__solver': 'liblinear'}


In [4]:
# printing the accuracy under each possible combination of parameters
import pandas as pd

df_results = pd.DataFrame(grid_search.cv_results_)

# Extract the columns of interest
df_pivot = df_results[['mean_test_score', 'param_model__C', 'param_model__penalty', 'param_model__solver']]

# Pivot the table
pivot_table = df_pivot.pivot_table(index='param_model__C', 
                                   columns=['param_model__penalty', 'param_model__solver'], 
                                   values='mean_test_score')

# Display the table with a neat format
with pd.option_context('display.float_format', '{:.3f}'.format):  # Displaying floats with 2 decimal places
    print(pivot_table)


param_model__penalty        l1              l2      
param_model__solver  liblinear  saga liblinear  saga
param_model__C                                      
0.001                    0.509 0.496     0.641 0.664
0.010                    0.509 0.509     0.671 0.669
0.100                    0.659 0.662     0.682 0.684
1.000                    0.680 0.680     0.688 0.688
10.000                   0.686 0.688     0.688 0.688
100.000                  0.688 0.688     0.688 0.688


The best parameters from grid search are 'model__C': 1, 'model__penalty': 'l2', 'model__solver': 'liblinear'. So this model is the better model. this model has better accuracy over other parameter values as obtained using grid search.

In [5]:
from sklearn.metrics import accuracy_score

#accuracy on test and train serts
train_preds = grid_search.predict(X_train)
test_preds = grid_search.predict(X_test)

train_acc = accuracy_score(y_train, train_preds)
test_acc = accuracy_score(y_test, test_preds)

print(f"Training accuracy: {train_acc}")
print(f"Test accuracy: {test_acc}")



Training accuracy: 0.71375
Test accuracy: 0.655
